# RunwayEye · antrenarea modelului YOLO (FOD)

Clase: `surub`, `piulita`, `cheie`, `metal`, `plastic` (exact aceste nume, fără diacritice).

1. **Runtime → Change runtime type → T4 GPU**.
2. Rulați celulele în ordine. Antrenarea durează ~20–40 de minute.
3. La final se descarcă `best.pt` și `metrics.json`. Copiați `best.pt` în `models/` din proiect.

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics roboflow

## Setul de date

**Varianta A (recomandată): direct din Roboflow.** Puneți cheia API în Colab: iconița 🔑 *Secrets* din stânga → `ROBOFLOW_API_KEY`. Completați workspace-ul, proiectul și versiunea (le vedeți în Roboflow la *Download Dataset → YOLOv8 → show download code*).

**Varianta B (fără cont Roboflow):** pe laptop, `python scripts/fod_to_yolo.py --voc <folder FOD-A> --extra <exportul Roboflow cu pozele machetei> --zip` face `data/dataset_fod.zip` (FOD-A, licență MIT, plus pozele noastre). Îl încărcați la celula următoare, cu `USE_ROBOFLOW_API = False`. Merge și un export Roboflow `.zip` în format **YOLOv8**.

In [ ]:
USE_ROBOFLOW_API = True
WORKSPACE = "workspace-ul-vostru"
PROJECT = "runwayeye-fod"
VERSION = 1

import glob, os, zipfile

if USE_ROBOFLOW_API:
    from google.colab import userdata
    from roboflow import Roboflow

    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
    DATA = os.path.join(dataset.location, "data.yaml")
else:
    from google.colab import files

    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("dataset")
    DATA = os.path.abspath(glob.glob("dataset/**/data.yaml", recursive=True)[0])

print("data.yaml:", DATA)
print(open(DATA).read())

In [ ]:
# Verificare: numele claselor trebuie să fie exact cele așteptate de aplicație.
import yaml

EXPECTED = {"surub", "piulita", "cheie", "metal", "plastic"}
names = yaml.safe_load(open(DATA))["names"]
names = set(names.values() if isinstance(names, dict) else names)
print("clase:", sorted(names))
if names != EXPECTED:
    print("ATENȚIE: clase diferite de", sorted(EXPECTED), "- aplicația afișează numele așa cum sunt")

## Antrenare

Model nano, imagini de 640 px. `patience` oprește antrenarea dacă nu mai progresează. Pentru un prim model rapid (~100 de poze), puneți `EPOCHS = 40`.

In [ ]:
from ultralytics import YOLO

EPOCHS = 80
model = YOLO("yolo11n.pt")  # pre-antrenat pe COCO; învățăm doar clasele noastre
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    patience=20,
    project="runs",
    name="fod",
    exist_ok=True,
)

## Rezultate (pentru slide)

mAP pe setul de **validare**, calculat de Ultralytics. Notați cifrele exact cum ies.

In [ ]:
import json

best = YOLO("runs/fod/weights/best.pt")
m = best.val(data=DATA, imgsz=640)
metrics = {
    "mAP50": round(float(m.box.map50), 4),
    "mAP50_95": round(float(m.box.map), 4),
    "precision": round(float(m.box.mp), 4),
    "recall": round(float(m.box.mr), 4),
    "per_class_mAP50_95": {best.names[int(c)]: round(float(v), 4) for c, v in zip(m.box.ap_class_index, m.box.maps[m.box.ap_class_index])},
    "epochs": EPOCHS,
    "imgsz": 640,
    "base_model": "yolo11n.pt",
}
json.dump(metrics, open("metrics.json", "w"), indent=2)
print(json.dumps(metrics, indent=2))

In [ ]:
# Imagini utile pentru prezentare: matricea de confuzie și curbele de antrenare.
from IPython.display import Image, display

for f in ["confusion_matrix_normalized.png", "results.png"]:
    p = f"runs/fod/{f}"
    if os.path.exists(p):
        display(Image(p, width=700))

In [ ]:
from google.colab import files

files.download("runs/fod/weights/best.pt")
files.download("metrics.json")
!zip -qj fod_plots.zip runs/fod/*.png
files.download("fod_plots.zip")